# COVID-19 India Data Analysis 🇮🇳

An end-to-end exploratory data analysis project using Python, Pandas, NumPy and Matplotlib. The notebook downloads a historical state-wise time series, cleans it, calculates key metrics and creates portfolio-ready visualisations.

## 1. Setup

The dataset is obtained from the archived COVID-19 India community data project. The source provides state-wise time-series data for confirmed, recovered and deceased cases. If the online file is unavailable, place a compatible CSV at `../data/covid_india.csv`.

In [ ]:
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
DATA_URL = 'https://raw.githubusercontent.com/amitvsavant/covid19-india-state-timeseries/master/data/covid19-india-statewise-timeseries.csv'
LOCAL_PATH = '../data/covid_india.csv'


## 2. Load the data

In [ ]:
def load_data():
    try:
        df = pd.read_csv(LOCAL_PATH)
        print(f'Loaded local dataset: {len(df):,} rows')
    except FileNotFoundError:
        response = requests.get(DATA_URL, timeout=30)
        response.raise_for_status()
        df = pd.read_csv(io.BytesIO(response.content))
        print(f'Loaded online dataset: {len(df):,} rows')
    df.columns = (df.columns.astype(str).str.strip().str.lower().str.replace(' ', '_', regex=False).str.replace('/', '_', regex=False))
    return df

df = load_data()
df.head()

## 3. Understand and clean the dataset

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
DATE_COL = 'date'
STATE_COL = 'state'
CONFIRMED_COL = 'total_confirmed_cases'
RECOVERED_COL = 'cured_discharged_migrated'
DEATHS_COL = 'death'

df[DATE_COL] = pd.to_datetime(df[DATE_COL], dayfirst=True, errors='coerce')
for col in [CONFIRMED_COL, RECOVERED_COL, DEATHS_COL]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
df = df.dropna(subset=[DATE_COL, STATE_COL]).sort_values(DATE_COL)
print(f'Clean rows: {len(df):,}')
df.tail()

## 4. India-wide trend

In [ ]:
daily = (df.groupby(DATE_COL, as_index=False)[[CONFIRMED_COL, RECOVERED_COL, DEATHS_COL]].sum().sort_values(DATE_COL))
daily['new_confirmed'] = daily[CONFIRMED_COL].diff().clip(lower=0).fillna(0)
daily['new_deaths'] = daily[DEATHS_COL].diff().clip(lower=0).fillna(0)
display(daily.tail())

plt.plot(daily[DATE_COL], daily[CONFIRMED_COL], label='Confirmed')
plt.plot(daily[DATE_COL], daily[RECOVERED_COL], label='Recovered')
plt.plot(daily[DATE_COL], daily[DEATHS_COL], label='Deaths')
plt.title('COVID-19 India: Cumulative Cases, Recoveries and Deaths')
plt.xlabel('Date'); plt.ylabel('People'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Daily case growth

In [ ]:
plt.plot(daily[DATE_COL], daily['new_confirmed'])
plt.title('Estimated Daily Increase in Confirmed Cases')
plt.xlabel('Date'); plt.ylabel('New confirmed cases'); plt.tight_layout(); plt.show()

peak_day = daily.loc[daily['new_confirmed'].idxmax()]
print('Highest estimated daily increase:', int(peak_day['new_confirmed']), 'on', peak_day[DATE_COL].date())

## 6. Top affected states/regions

In [ ]:
latest_date = df[DATE_COL].max()
latest = df[df[DATE_COL] == latest_date].sort_values(CONFIRMED_COL, ascending=False)
top10 = latest.head(10).sort_values(CONFIRMED_COL)
display(latest.head(10)[[STATE_COL, CONFIRMED_COL, RECOVERED_COL, DEATHS_COL]])

plt.barh(top10[STATE_COL], top10[CONFIRMED_COL])
plt.title(f'Top 10 States/Regions by Confirmed Cases ({latest_date.date()})')
plt.xlabel('Confirmed cases'); plt.tight_layout(); plt.show()

## 7. Case-fatality and recovery rates

In [ ]:
latest_metrics = latest.copy()
latest_metrics['recovery_rate_%'] = np.where(latest_metrics[CONFIRMED_COL] > 0, latest_metrics[RECOVERED_COL] / latest_metrics[CONFIRMED_COL] * 100, np.nan)
latest_metrics['fatality_rate_%'] = np.where(latest_metrics[CONFIRMED_COL] > 0, latest_metrics[DEATHS_COL] / latest_metrics[CONFIRMED_COL] * 100, np.nan)
display(latest_metrics[[STATE_COL, CONFIRMED_COL, RECOVERED_COL, DEATHS_COL, 'recovery_rate_%', 'fatality_rate_%']].head(15))

## 8. Key findings to discuss

1. The pandemic produced distinct periods of rapid case growth in India.
2. The distribution of reported cases was highly uneven across states and regions.
3. Cumulative curves are useful for understanding overall burden, while daily changes show short-term acceleration.
4. Reporting delays, retrospective revisions and changes in definitions mean the figures should be interpreted as reported historical data rather than perfect real-time measurements.

### Portfolio note
This notebook demonstrates data acquisition, cleaning, feature calculation, exploratory analysis and visualisation — all useful skills for a junior data analyst/data scientist portfolio.

**Disclaimer:** Educational project only; not for medical or policy decisions.